# Fink LSST Data Transfer — Light Curve Explorer

This notebook provides enhanced light curve visualisation and population diagnostics
for alerts downloaded from the [Fink Data Transfer Service](https://lsst.fink-portal.org/download).

## Photometric data model (LSST / Fink)

All fluxes are on the **difference image** (science − template), in nJy.

| Table | Content | Flux field used |
|---|---|---|
| `diaSource` + `prvDiaSources` | Detections that **triggered** an alert | `psfFlux` (diff PSF) |
| `prvDiaForcedSources` | Forced-position measurements on visits that did **not** trigger an alert (pre/post-detection baseline) | `psfFlux` (diff PSF, open symbols) |

Both tables carry `scienceFlux` = `psfFlux` + `templateFlux` (science image flux).  
The relation `scienceFlux = psfFlux + templateFlux` holds in both; `templateFlux` is absent from `prvDiaForcedSources`.

**Author:** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS (Université Paris-Saclay)  
**Creation date:** 2026-05-06  
**Data:** `ftransfer_lsst_2026-05-06_231893`

## 0. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from collections import defaultdict

from fink_client.visualisation import extract_field, show_stamps

warnings.filterwarnings("ignore")

# ── LSST band palette (wavelength-ordered, blue → red) ────────────────────────
BAND_COLORS = {
    "u": "#5B4FCF",
    "g": "#1E90FF",
    "r": "#2ECC40",
    "i": "#FF851B",
    "z": "#E74C3C",
    "y": "#8B0000",
}
BAND_MARKERS = {"u": "o", "g": "s", "r": "D", "i": "^", "z": "v", "y": "P"}
ALL_BANDS = ["u", "g", "r", "i", "z", "y"]

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)
print("Imports OK")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

## 1. Load the parquet dataset

In [ ]:
TOPICNAME = "ftransfer_lsst_2026-05-06_231893"
SCHEMA_FILE = f"arrow_schema_{TOPICNAME}.metadata"

schema = pq.read_schema(SCHEMA_FILE)
table = pq.read_table(TOPICNAME, schema=schema)
pdf = table.to_pandas()

print(f"Total alerts loaded : {len(pdf):,}")
print(f"Columns             : {list(pdf.columns)}")

## 2. Inspect available flux fields in prvDiaForcedSources

Quick sanity check: confirm which flux fields are actually populated.

In [ ]:
# Inspect one alert that has prvDiaForcedSources
# prvDiaForcedSources is a Python list (or None) in each row.
# After pd.read_parquet it can be: None, [], or a list of dicts.
# NEVER do `if forced:` on it — ambiguous if it's an array.

for _, row in pdf.iterrows():
    forced = row["prvDiaForcedSources"]  # direct access, not .get()
    if forced is None:
        continue
    try:
        n = len(forced)
    except TypeError:
        continue
    if n == 0:
        continue
    first = forced[0]
    if first is None:
        continue
    print("Fields in prvDiaForcedSources[0]:")
    if isinstance(first, dict):
        print(list(first.keys()))
        print("\nSample psfFlux    :", first.get("psfFlux"))
        print("Sample scienceFlux :", first.get("scienceFlux"))
    else:
        print(type(first), first)
    break

## 3. Build a per-DiaObject summary catalogue

Multiple parquet rows share the same `diaObjectId`. We keep the **most recent alert**
per object (largest MJD in `diaSource.midpointMjdTai`) so that `prvDiaSources` already
contains the longest available history.

In [ ]:
def build_object_catalog(pdf):
    """Return one summary row per diaObject, keeping the latest alert."""
    records = []
    grouped = defaultdict(list)

    for idx, row in pdf.iterrows():
        do = row.get("diaObject")
        oid = do.get("diaObjectId") if isinstance(do, dict) else None
        if oid is None:
            continue
        ds = row.get("diaSource")
        mjd = ds.get("midpointMjdTai") if isinstance(ds, dict) else None
        grouped[oid].append((mjd if mjd is not None else -np.inf, idx))

    for oid, entries in grouped.items():
        entries.sort(key=lambda x: x[0])
        _, best_idx = entries[-1]
        row = pdf.loc[best_idx]
        do = row.get("diaObject") or {}
        xm = row.get("xm") or {}
        clf = row.get("clf") or {}
        pred = row.get("pred") or {}

        records.append(
            {
                "diaObjectId": oid,
                "ra": do.get("ra"),
                "dec": do.get("dec"),
                "target_name": row.get("target_name"),
                "tns_fullname": xm.get("tns_fullname"),
                "tns_type": xm.get("tns_type"),
                "tns_type_recomputed": row.get("tns_type_recomputed"),
                "tns_z": xm.get("tns_redshift"),
                "zphot": xm.get("legacydr8_zphot"),
                "zphot_err": xm.get("legacydr8_e_zphot"),
                "simbad_otype": xm.get("simbad_otype"),
                "snn_score": clf.get("snnSnVsOthers_score"),
                "cats_class": clf.get("cats_class"),
                "cats_score": clf.get("cats_score"),
                "earlySNIa_score": clf.get("earlySNIa_score"),
                "pred_xm": pred.get("main_label_crossmatch"),
                "pred_clf": pred.get("main_label_classifier"),
                "is_sso": pred.get("is_sso"),
                "n_alerts": len(entries),
                "latest_idx": best_idx,
            }
        )

    return pd.DataFrame(records)


cat = build_object_catalog(pdf)
print(f"Unique diaObjects : {len(cat):,}")
cat.head()

## 4. Population diagnostics

### 4.1 Source type breakdown

In [ ]:
def make_type_label(row):
    """Best available classification label for one object row."""
    for col in ("tns_type_recomputed", "tns_type", "pred_xm", "simbad_otype"):
        v = row.get(col) if isinstance(row, dict) else getattr(row, col, None)
        if v and str(v).strip() not in ("", "None", "nan", "Unknown"):
            return str(v).strip()
    return "Unknown"


cat["type_label"] = cat.apply(make_type_label, axis=1)
type_counts = cat["type_label"].value_counts()

fig, ax = plt.subplots(figsize=(11, 4))
colors = plt.cm.tab20(np.linspace(0, 1, len(type_counts)))
bars = ax.barh(
    type_counts.index[::-1], type_counts.values[::-1], color=colors[::-1], edgecolor="k", linewidth=0.4
)
for bar, val in zip(bars, type_counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, str(val), va="center", fontsize=9)
ax.set_xlabel("Number of unique diaObjects")
ax.set_title("Source types in the transfer (best available label)")
ax.set_xscale("log")
plt.tight_layout()
plt.savefig("diag_source_types.pdf", bbox_inches="tight")
plt.show()
print(type_counts.to_string())

### 4.2 Redshift distributions by source type

In [ ]:
cat_z = cat.copy()
cat_z["z_best"] = cat_z["tns_z"].where(cat_z["tns_z"].notna(), cat_z["zphot"])
cat_z["z_source"] = "photo-z (LegacyDR8)"
cat_z.loc[cat_z["tns_z"].notna(), "z_source"] = "spec-z (TNS)"

has_z = cat_z[cat_z["z_best"].notna() & (cat_z["z_best"].astype(float) > 0)]
print(f"Objects with a redshift : {len(has_z)} / {len(cat_z)}")
print(f"  spec-z (TNS) : {(has_z['z_source'] == 'spec-z (TNS)').sum()}")
print(f"  photo-z      : {(has_z['z_source'] == 'photo-z (LegacyDR8)').sum()}")

In [ ]:
top_types = type_counts[type_counts >= 2].index.tolist()
has_z = has_z.copy()
has_z["type_plot"] = has_z["type_label"].where(has_z["type_label"].isin(top_types), other="Other")

type_groups = has_z.groupby("type_plot")
n_types = len(type_groups)
ncols = min(3, n_types)
nrows = int(np.ceil(n_types / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), sharey=False)
axes = np.array(axes).flatten()
zbins = np.linspace(0, 1.5, 31)

for ax_i, (ttype, grp) in enumerate(sorted(type_groups, key=lambda x: -len(x[1]))):
    ax = axes[ax_i]
    spec = grp[grp["z_source"] == "spec-z (TNS)"]["z_best"].dropna().astype(float)
    photo = grp[grp["z_source"] == "photo-z (LegacyDR8)"]["z_best"].dropna().astype(float)
    if len(spec):
        ax.hist(
            spec,
            bins=zbins,
            color="steelblue",
            alpha=0.85,
            label=f"spec-z (N={len(spec)})",
            edgecolor="k",
            lw=0.3,
        )
    if len(photo):
        ax.hist(
            photo,
            bins=zbins,
            color="tomato",
            alpha=0.6,
            label=f"photo-z (N={len(photo)})",
            edgecolor="k",
            lw=0.3,
        )
    ax.set_title(ttype, fontweight="bold")
    ax.set_xlabel("Redshift")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

for ax in axes[n_types:]:
    ax.set_visible(False)

plt.suptitle("Redshift distributions by source type", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("diag_redshift_by_type.pdf", bbox_inches="tight")
plt.show()

### 4.3 Classifier score distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
score_configs = [
    ("snn_score", "SNN (SN vs Others)", "royalblue"),
    ("cats_score", "CATS score", "seagreen"),
    ("earlySNIa_score", "Early SNIa score", "firebrick"),
]
for ax, (col, title, color) in zip(axes, score_configs):
    vals = pd.to_numeric(cat[col], errors="coerce").dropna()
    if len(vals):
        ax.hist(vals, bins=30, color=color, alpha=0.8, edgecolor="k", linewidth=0.4)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)
    ax.axvline(0.5, color="k", ls="--", lw=1.2, label="threshold=0.5")
    ax.legend(fontsize=8)
plt.suptitle("Fink classifier score distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_classifier_scores.pdf", bbox_inches="tight")
plt.show()

## 5. Enhanced per-object light curves

### Photometric conventions used in the plots

```
● filled symbols  →  diaSource + prvDiaSources   (detections, psfFlux diff)
○ open symbols    →  prvDiaForcedSources          (forced non-detections, psfFlux diff)
```

Both use `psfFlux` on the **difference image** — they share the same flux axis and
can be overplotted directly.  The forced points provide the pre- and post-transient
baseline and are expected to scatter around zero.

An optional second panel shows `scienceFlux` from both tables, which represents
the absolute flux on the science image (`scienceFlux = psfFlux + templateFlux`).

In [ ]:
def make_lc_title(row):
    """Build an informative figure title."""
    oid = row["diaObjectId"]
    tns = row.get("tns_fullname") or ""
    ttype = row.get("tns_type_recomputed") or row.get("tns_type") or "?"
    z_spec = row.get("tns_z")
    z_phot = row.get("zphot")
    z_phot_e = row.get("zphot_err")

    z_str = ""
    if z_spec is not None:
        try:
            if not np.isnan(float(z_spec)):
                z_str = f"$z_{{\\rm spec}}={float(z_spec):.4f}$"
        except (TypeError, ValueError):
            pass
    if not z_str and z_phot is not None:
        try:
            if not np.isnan(float(z_phot)):
                ze = f"\u00b1{float(z_phot_e):.3f}" if z_phot_e is not None else ""
                z_str = f"$z_{{\\rm phot}}={float(z_phot):.3f}{ze}$"
        except (TypeError, ValueError):
            pass

    def _fmt_score(name, val):
        try:
            return f"{name}={float(val):.2f}" if val is not None and not np.isnan(float(val)) else ""
        except (TypeError, ValueError):
            return ""

    snn_str = _fmt_score("SNN", row.get("snn_score"))
    esnia_str = _fmt_score("EarlySNIa", row.get("earlySNIa_score"))

    parts = [f"ID {oid}"]
    if tns:
        parts.append(f"TNS: {tns}")
    parts.append(f"type: {ttype}")
    if z_str:
        parts.append(z_str)
    clf_parts = [s for s in [snn_str, esnia_str] if s]
    if clf_parts:
        parts.append(" | ".join(clf_parts))
    return "   ·   ".join(parts)


print("make_lc_title() defined.")

In [ ]:
def plot_object_lc(row, pdf, save_prefix=None, show_science=False):
    """Plot the complete light curve for one diaObject.

    Layout (single flux axis by default)
    ─────────────────────────────────────
    Single panel:
      ● filled   = psfFlux from diaSource + prvDiaSources   (detections)
      ○ open     = psfFlux from prvDiaForcedSources          (forced non-detections)
    Both are difference-image fluxes → directly comparable on the same axis.

    Optional second panel (show_science=True):
      scienceFlux = psfFlux + templateFlux (science-image absolute flux)
      from both diaSource/prvDiaSources (filled) and prvDiaForcedSources (open).

    Parameters
    ----------
    row          : pd.Series from the object catalogue
    pdf          : full alert DataFrame
    save_prefix  : str or None — if given, save to f'{save_prefix}_{diaObjectId}.pdf'
    show_science : bool — add a second panel with scienceFlux
    """
    oid = row["diaObjectId"]

    # Collect all alerts for this object
    all_alerts = []
    for _, r in pdf.iterrows():
        do = r.get("diaObject")
        rid = do.get("diaObjectId") if isinstance(do, dict) else None
        if rid == oid:
            all_alerts.append(r)
    if not all_alerts:
        print(f"No alerts found for diaObjectId={oid}")
        return

    # Use the latest alert for the longest prvDiaSources
    alert = max(
        all_alerts,
        key=lambda r: (
            (r["diaSource"].get("midpointMjdTai") or -np.inf) if isinstance(r["diaSource"], dict) else -np.inf
        ),
    )

    # ── Difference-image psfFlux: detections ──────────────────────────────────
    mjd_det = extract_field(alert, "midpointMjdTai", current="diaSource", previous="prvDiaSources")
    flux_det = extract_field(alert, "psfFlux", current="diaSource", previous="prvDiaSources")
    fluxe_det = extract_field(alert, "psfFluxErr", current="diaSource", previous="prvDiaSources")
    band_det = extract_field(alert, "band", current="diaSource", previous="prvDiaSources")

    # ── Difference-image psfFlux: forced non-detections ───────────────────────
    # prvDiaForcedSources also carries psfFlux (diff-image forced PSF flux)
    # These are visits that did NOT trigger an alert — same flux type, open symbols
    mjd_forced = extract_field(alert, "midpointMjdTai", current="diaSource", previous="prvDiaForcedSources")
    flux_forced = extract_field(alert, "psfFlux", current="diaSource", previous="prvDiaForcedSources")
    fluxe_forced = extract_field(alert, "psfFluxErr", current="diaSource", previous="prvDiaForcedSources")
    band_forced = extract_field(alert, "band", current="diaSource", previous="prvDiaForcedSources")

    # ── Science-image flux (optional panel) ───────────────────────────────────
    if show_science:
        sciflux_det = extract_field(alert, "scienceFlux", current="diaSource", previous="prvDiaSources")
        scifluxe_det = extract_field(alert, "scienceFluxErr", current="diaSource", previous="prvDiaSources")
        sciflux_frc = extract_field(alert, "scienceFlux", current="diaSource", previous="prvDiaForcedSources")
        scifluxe_frc = extract_field(
            alert, "scienceFluxErr", current="diaSource", previous="prvDiaForcedSources"
        )

    title = make_lc_title(row)

    # ── Figure ────────────────────────────────────────────────────────────────
    nrows_fig = 2 if show_science else 1
    fig = plt.figure(figsize=(13, 4.5 * nrows_fig + 0.5))
    gs = gridspec.GridSpec(nrows_fig, 1, hspace=0.08)
    ax_diff = fig.add_subplot(gs[0])
    if show_science:
        ax_sci = fig.add_subplot(gs[1], sharex=ax_diff)

    # ── Panel 1: difference psfFlux ───────────────────────────────────────────
    # Forced (open) drawn first (background), detections (filled) on top
    has_forced_data = mjd_forced is not None and len(mjd_forced) > 1
    has_det_data = mjd_det is not None and len(mjd_det) > 0

    legend_handles = []

    for band in ALL_BANDS:
        c = BAND_COLORS[band]
        m = BAND_MARKERS[band]

        # --- forced (open symbols, smaller, lighter) ---
        if has_forced_data:
            mask_f = ~pd.isna(fluxe_forced) & (band_forced == band)
            if mask_f.sum():
                ax_diff.errorbar(
                    mjd_forced[mask_f],
                    flux_forced[mask_f],
                    fluxe_forced[mask_f],
                    fmt=m,
                    color=c,
                    ms=5,
                    lw=0.8,
                    capsize=1.5,
                    alpha=0.5,
                    mfc="none",
                    mew=1.2,  # open markers
                    zorder=2,
                )

        # --- detections (filled symbols, larger) ---
        if has_det_data:
            mask_d = ~pd.isna(fluxe_det) & (band_det == band)
            if mask_d.sum():
                h = ax_diff.errorbar(
                    mjd_det[mask_d],
                    flux_det[mask_d],
                    fluxe_det[mask_d],
                    fmt=m,
                    color=c,
                    ms=7,
                    lw=1.2,
                    capsize=2,
                    alpha=0.9,
                    mfc=c,  # filled markers
                    label=f"{band}",
                    zorder=3,
                )
                legend_handles.append(h)

    ax_diff.axhline(0, color="grey", ls="--", lw=0.8, alpha=0.6)
    ax_diff.set_ylabel("Diff. flux  [nJy]", fontsize=11)
    ax_diff.set_title(title, fontsize=10, pad=6, fontweight="bold")
    ax_diff.grid(alpha=0.3)

    # Manual legend: add proxy for forced points
    from matplotlib.lines import Line2D

    proxy_forced = Line2D(
        [0], [0], marker="o", color="grey", ms=5, ls="", mfc="none", mew=1.2, label="forced (non-det.)"
    )
    proxy_det = Line2D([0], [0], marker="o", color="grey", ms=7, ls="", mfc="grey", label="detection")
    handles_extra = [proxy_det, proxy_forced]
    ax_diff.legend(handles=handles_extra + legend_handles, loc="best", ncol=4, framealpha=0.8, fontsize=9)

    if show_science:
        plt.setp(ax_diff.get_xticklabels(), visible=False)
        # ── Panel 2: science flux ─────────────────────────────────────────────
        for band in ALL_BANDS:
            c = BAND_COLORS[band]
            m = BAND_MARKERS[band]
            if has_forced_data and sciflux_frc is not None:
                mask_f = ~pd.isna(scifluxe_frc) & (band_forced == band)
                if mask_f.sum():
                    ax_sci.errorbar(
                        mjd_forced[mask_f],
                        sciflux_frc[mask_f],
                        scifluxe_frc[mask_f],
                        fmt=m,
                        color=c,
                        ms=5,
                        lw=0.8,
                        capsize=1.5,
                        alpha=0.5,
                        mfc="none",
                        mew=1.2,
                        zorder=2,
                    )
            if has_det_data and sciflux_det is not None:
                mask_d = ~pd.isna(scifluxe_det) & (band_det == band)
                if mask_d.sum():
                    ax_sci.errorbar(
                        mjd_det[mask_d],
                        sciflux_det[mask_d],
                        scifluxe_det[mask_d],
                        fmt=m,
                        color=c,
                        ms=7,
                        lw=1.2,
                        capsize=2,
                        alpha=0.9,
                        mfc=c,
                        zorder=3,
                    )
        ax_sci.axhline(0, color="grey", ls="--", lw=0.8, alpha=0.6)
        ax_sci.set_ylabel("Science flux  [nJy]", fontsize=10)
        ax_sci.set_xlabel("MJD", fontsize=11)
        ax_sci.grid(alpha=0.3)
    else:
        ax_diff.set_xlabel("MJD", fontsize=11)

    plt.tight_layout()
    if save_prefix:
        plt.savefig(f"{save_prefix}_{oid}.pdf", bbox_inches="tight")
    plt.show()
    # plt.close()


print("plot_object_lc() defined.")

### 5.1 Plot all objects (loop over diaObjects)

Control parameters:
- `MAX_OBJECTS` — limit number of plots (`None` = all)
- `SHOW_SCIENCE` — add a second panel with `scienceFlux`  
- `SAVE_FIGS`   — write individual PDFs

In [ ]:
MAX_OBJECTS = 100
SHOW_SCIENCE = True  # set True to see the scienceFlux panel
SAVE_FIGS = False

subset = cat.head(MAX_OBJECTS) if MAX_OBJECTS else cat
save_pfx = "lc" if SAVE_FIGS else None

for _, row in subset.iterrows():
    plot_object_lc(row, pdf, save_prefix=save_pfx, show_science=SHOW_SCIENCE)

### 5.2 Objects with a spectroscopic redshift (TNS)

In [ ]:
cat_specz = cat[cat["tns_z"].notna() & (pd.to_numeric(cat["tns_z"], errors="coerce") > 0)].copy()
print(f"Objects with TNS spec-z : {len(cat_specz)}")

for _, row in cat_specz.iterrows():
    plot_object_lc(row, pdf, show_science=False)

### 5.3 High-confidence SN Ia candidates (SNN > 0.9)

In [ ]:
cat_snia = cat[pd.to_numeric(cat["snn_score"], errors="coerce") > 0.9].copy()
print(f"High-confidence SN Ia candidates (SNN > 0.9) : {len(cat_snia)}")

for _, row in cat_snia.iterrows():
    plot_object_lc(row, pdf, show_science=True)  # show science flux for SNe

## 6. Summary table

In [ ]:
summary_cols = [
    "diaObjectId",
    "target_name",
    "tns_fullname",
    "type_label",
    "tns_z",
    "zphot",
    "snn_score",
    "earlySNIa_score",
    "ra",
    "dec",
    "n_alerts",
]
display_cols = [c for c in summary_cols if c in cat.columns]
summary = cat[display_cols].sort_values("snn_score", ascending=False)

summary.to_csv("fink_object_catalog.csv", index=False)
print(f"Saved fink_object_catalog.csv ({len(summary)} objects)")

try:
    from IPython.display import display

    display(summary.head(30))
except ImportError:
    print(summary.head(30).to_string())